In [ ]:
import sys
from pathlib import Path

HICOLA_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'scripts' / 'hicola_paths.py').exists())
sys.path.insert(0, str(HICOLA_ROOT / 'scripts'))
import hicola_paths as hp

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import fitsio

In [ ]:
def load_power_spectrum(file):
    
    table = pd.read_csv(file,
    sep=r"\s+",
    comment="#",
    names=["k (h/Mpc)", "P_k (Mpc/h)^3", "P_linear_k (Mpc/h)^3"])

    k = table["k (h/Mpc)"]
    Pk = table["P_k (Mpc/h)^3"]
    
    return k, Pk

In [ ]:
def plot_power_spectrum(k, Pk):

    fig, ax = plt.subplots(figsize=(6.5, 5.5), dpi=300)

    ax.loglog(k, Pk, lw=2)

    ax.set_xlabel(r"$k \; [h/\mathrm{Mpc}]$", fontsize=14)
    ax.set_ylabel(r"$P(k)[(\mathrm{Mpc}/h)^3]$", fontsize=14)

    ax.tick_params(axis='both', which='major', labelsize=12)
    ax.tick_params(axis='both', which='minor', labelsize=10)

    plt.tight_layout()
    plt.savefig("fig/power_spectrum.png")
    plt.show()

In [ ]:
def compare_power_spectrum(ks, Pks, labels, title=None):

    fig, ax = plt.subplots(figsize=(6.5, 5.5), dpi=300)

    for i in range(len(ks)):
        ax.loglog(ks[i], Pks[i], lw=2, label=labels[i])
    
    if title:
        ax.set_title(title, fontsize=14)

    ax.set_xlabel(r"$k \; [h/\mathrm{Mpc}]$", fontsize=14)
    ax.set_ylabel(r"$P(k)[(\mathrm{Mpc}/h)^3]$", fontsize=14)

    ax.tick_params(axis='both', which='major', labelsize=12)
    ax.tick_params(axis='both', which='minor', labelsize=10)

    plt.tight_layout()
    plt.legend()
    plt.savefig("fig/power_spectrum.png")
    plt.show()

In [ ]:
def ratio_power_spectrum(k1, Pk1, k2, Pk2, labels):

    fig, ax = plt.subplots(figsize=(8.5, 5.5), dpi=300)

    ax.plot(k1, Pk2/Pk1, lw=2)

    ax.set_xlabel(r"$k \; [h/\mathrm{Mpc}]$", fontsize=14)
    ax.set_ylabel(r"$p^{\mathrm{" + labels[1] + r"}}(k,z=0)/p^{\mathrm{" + labels[0] + r"}}(k,z=0)$", fontsize=14)
    ax.set_xscale('log')
    #ax.set_xlim(2e-1,2)

    ax.tick_params(axis='both', which='major', labelsize=12)
    ax.tick_params(axis='both', which='minor', labelsize=10)

    plt.tight_layout()
    plt.savefig("fig/ratio_power_spectrum.png")
    plt.show()

### Hi-COLA paper reproduction

In [ ]:
k_LCDM, Pk_LCDM = load_power_spectrum('LCDM_test/snapshot_LCDM_test_z0.000/pofk_z0.000.txt')
k_cuGal, Pk_cuGal = load_power_spectrum('cuGal_test/snapshot_cuGal_test_z0.000/pofk_z0.000.txt')
k_nDGP, Pk_nDGP = load_power_spectrum('nDGP_model/snapshot_nDGP_model_z0.000/pofk_z0.000.txt')

In [ ]:
labels = ['LCDM', 'cuGal','nDGP']
compare_power_spectrum([k_LCDM,k_cuGal,k_nDGP], [Pk_LCDM,Pk_cuGal,Pk_nDGP], labels)

In [ ]:
labels = ['LCDM', 'cuGal']
ratio_power_spectrum(k_LCDM,Pk_LCDM,k_cuGal,Pk_cuGal,labels)

In [ ]:
labels = ['LCDM', 'nDGP']
ratio_power_spectrum(k_LCDM,Pk_LCDM,k_nDGP,Pk_nDGP,labels)

### Bart's ELEPHANT simulation

In [ ]:
k_LCDM, Pk_LCDM = load_power_spectrum(hp.reference_file('COLA_NM4096', 'LCDM', 1, 1.65, 'PS'))
k_nDGP, Pk_nDGP = load_power_spectrum(hp.reference_file('COLA_NM4096', 'nDGP_N1.2', 1, 1.65, 'PS'))

In [ ]:
labels = ['LCDM','nDGP']
compare_power_spectrum([k_LCDM,k_nDGP], [Pk_LCDM,Pk_nDGP], labels)

In [ ]:
labels = ['LCDM', 'nDGP']
ratio_power_spectrum(k_LCDM,Pk_LCDM,k_nDGP,Pk_nDGP,labels)

### Our COLA simulation

In [ ]:
k_LCDM, Pk_LCDM = load_power_spectrum(hp.standalone_run_dir('LCDM') / 'snapshot_LCDM_z1.650' / 'pofk_z1.650.txt')
k_nDGP, Pk_nDGP = load_power_spectrum(hp.standalone_run_dir('nDGP') / 'snapshot_nDGP_z1.650' / 'pofk_z1.650.txt')

In [ ]:
labels = ['LCDM','nDGP']
compare_power_spectrum([k_LCDM,k_nDGP], [Pk_LCDM,Pk_nDGP], labels)

In [ ]:
labels = ['LCDM', 'nDGP']
ratio_power_spectrum(k_LCDM,Pk_LCDM,k_nDGP,Pk_nDGP,labels)

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

pairs = [(0,'1.650'), (1,'1.400'), (2,'1.200'), (3,'1.000')]
for x,z in pairs:
    try:
        k_l, Pk_l = load_power_spectrum(hp.reference_file('COLA_NM4096', 'LCDM', 1, float(z), 'PS'))
        k_n, Pk_n = load_power_spectrum(hp.reference_file('COLA_NM4096', 'nDGP_N1.2', 1, float(z), 'PS'))
    except Exception as e:
        print(f'Could not load Bart snap files for x={x}:', e)
        continue
    try:
        kpl, Pkpl = load_power_spectrum(hp.standalone_run_dir('LCDM') / f'snapshot_LCDM_z{z}' / f'pofk_z{z}.txt')
        kpn, Pkpn = load_power_spectrum(hp.standalone_run_dir('nDGP') / f'snapshot_nDGP_z{z}' / f'pofk_z{z}.txt')
    except Exception as e:
        print(f'Could not load pofk snapshot files for z={z}:', e)
        continue
    fig, ax = plt.subplots(figsize=(7,5))
    ax.plot(k_l, (Pk_n/ Pk_l), label='COLA_NM4096', lw=2)
    ax.plot(kpl, (Pkpn/ Pkpl), label='Our simulation', lw=2, ls='--')
    ax.set_xscale('log')
    ax.set_xlabel(r'$k \, [h/\mathrm{Mpc}]$')
    ax.set_ylabel('P_n/P_L')
    ax.set_title(f'Power spectrum ratio comparison at z={z}')
    ax.legend()
    ax.grid(True, which='both', ls=':', alpha=0.6)
    plt.tight_layout()
    plt.show()


### Comparison

In [ ]:
# Choose one of the COLA_NM4096 reference boxes (1 ... 10)
box_n = 1

# Load the reference Bart data for one chosen box
k_LCDM_Bart, Pk_LCDM_Bart = load_power_spectrum(hp.reference_file('COLA_NM4096', 'LCDM', box_n, 1.65, 'PS'))
k_nDGP_Bart, Pk_nDGP_Bart = load_power_spectrum(hp.reference_file('COLA_NM4096', 'nDGP_N1.2', box_n, 1.65, 'PS'))

# Load our simulation results at z=1.650
k_LCDM_Me, Pk_LCDM_Me = load_power_spectrum(hp.standalone_run_dir('LCDM') / 'snapshot_LCDM_z1.650' / 'pofk_z1.650.txt')
k_nDGP_Me, Pk_nDGP_Me = load_power_spectrum(hp.standalone_run_dir('nDGP') / 'snapshot_nDGP_z1.650' / 'pofk_z1.650.txt')

In [ ]:
labels = ['COLA_NM4096','Our simulation']
compare_power_spectrum([k_LCDM_Bart,k_LCDM_Me], [Pk_LCDM_Bart,Pk_LCDM_Me], labels, title=f'LCDM Power Spectrum Comparison (Box {box_n}) at z=1.65')

In [ ]:
labels = ['COLA_NM4096','Our simulation']
compare_power_spectrum([k_nDGP_Bart,k_nDGP_Me], [Pk_nDGP_Bart,Pk_nDGP_Me], labels, title=f'nDGP Power Spectrum Comparison (Box {box_n}) at z=1.65')

In [ ]:
# Helper functions for averaging over multiple Box iterations

def load_bart_box_power_spectra(box_n):
    k_LCDM, Pk_LCDM = load_power_spectrum(hp.reference_file('COLA_NM4096', 'LCDM', box_n, 1.65, 'PS'))
    k_nDGP, Pk_nDGP = load_power_spectrum(hp.reference_file('COLA_NM4096', 'nDGP_N1.2', box_n, 1.65, 'PS'))
    return k_LCDM, Pk_LCDM, k_nDGP, Pk_nDGP


def mean_std_over_boxes(Pk_list):
    stack = np.vstack([np.asarray(pk) for pk in Pk_list])
    return np.mean(stack, axis=0), np.std(stack, axis=0)

# Average over the COLA_NM4096 reference boxes 1..5
n_boxes = 5
k_LCDM_boxes = []
Pk_LCDM_boxes = []
k_nDGP_boxes = []
Pk_nDGP_boxes = []
for i in range(1, n_boxes + 1):
    k_l, Pk_l, k_n, Pk_n = load_bart_box_power_spectra(i)
    k_LCDM_boxes.append(k_l)
    Pk_LCDM_boxes.append(Pk_l)
    k_nDGP_boxes.append(k_n)
    Pk_nDGP_boxes.append(Pk_n)

k_LCDM_avg, Pk_LCDM_avg, Pk_LCDM_std = k_LCDM_boxes[0], *mean_std_over_boxes(Pk_LCDM_boxes)
k_nDGP_avg, Pk_nDGP_avg, Pk_nDGP_std = k_nDGP_boxes[0], *mean_std_over_boxes(Pk_nDGP_boxes)

# Plot average with variance shading
fig, ax = plt.subplots(figsize=(6.5, 5.5), dpi=300)
ax.loglog(k_LCDM_avg, Pk_LCDM_avg, lw=2, label='COLA_NM4096 mean')
ax.fill_between(k_LCDM_avg, np.maximum(Pk_LCDM_avg - Pk_LCDM_std, 1e-20), Pk_LCDM_avg + Pk_LCDM_std,
                color='tab:orange', alpha=0.25, label='1σ across boxes')
ax.loglog(k_LCDM_Me, Pk_LCDM_Me, lw=2, ls='--', label='Our simulation')
ax.set_title('LCDM at z=1.65', fontsize=14)
ax.set_xlabel(r'$k \; [h/\mathrm{Mpc}]$', fontsize=14)
ax.set_ylabel(r'$P(k)[(\mathrm{Mpc}/h)^3]$', fontsize=14)
ax.tick_params(axis='both', which='major', labelsize=12)
ax.tick_params(axis='both', which='minor', labelsize=10)
ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6.5, 5.5), dpi=300)
ax.loglog(k_nDGP_avg, Pk_nDGP_avg, lw=2, label='COLA_NM4096 mean')
ax.fill_between(k_nDGP_avg, np.maximum(Pk_nDGP_avg - Pk_nDGP_std, 1e-20), Pk_nDGP_avg + Pk_nDGP_std,
                color='tab:orange', alpha=0.25, label='1σ across boxes')
ax.loglog(k_nDGP_Me, Pk_nDGP_Me, lw=2, ls='--', label='Our simulation')
ax.set_title('nDGP at z=1.65', fontsize=14)
ax.set_xlabel(r'$k \; [h/\mathrm{Mpc}]$', fontsize=14)
ax.set_ylabel(r'$P(k)[(\mathrm{Mpc}/h)^3]$', fontsize=14)
ax.tick_params(axis='both', which='major', labelsize=12)
ax.tick_params(axis='both', which='minor', labelsize=10)
ax.legend()
plt.tight_layout()
plt.show()

# Interpolate our simulation to the COLA mean k grid before normalizing
Pk_LCDM_Me_interp = np.interp(k_LCDM_avg, k_LCDM_Me, Pk_LCDM_Me)
Pk_nDGP_Me_interp = np.interp(k_nDGP_avg, k_nDGP_Me, Pk_nDGP_Me)

fig, ax = plt.subplots(figsize=(6.5, 5.5), dpi=300)
ax.semilogx(k_LCDM_avg, (Pk_LCDM_Me_interp - Pk_LCDM_avg) / Pk_LCDM_std, lw=2, label='LCDM (Our - mean) / std')
ax.semilogx(k_nDGP_avg, (Pk_nDGP_Me_interp - Pk_nDGP_avg) / Pk_nDGP_std, lw=2, ls='--', label='nDGP (Our - mean) / std')
ax.axhline(0, color='k', lw=1, ls=':')
#ax.set_title('Normalized Power Spectrum Residuals at z=1.65', fontsize=14)
ax.set_xlabel(r'$k \; [h/\mathrm{Mpc}]$', fontsize=14)
ax.set_ylabel(r'$(P_{\mathrm{our}}-P_{\mathrm{mean}})/\sigma$', fontsize=14)
ax.tick_params(axis='both', which='major', labelsize=12)
ax.tick_params(axis='both', which='minor', labelsize=10)
ax.legend()
ax.grid(True, which='both', ls=':', alpha=0.4)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6.5, 5.5), dpi=300)
ax.semilogx(k_LCDM_avg, Pk_LCDM_Me_interp - Pk_LCDM_avg, lw=2, label='LCDM (Our - mean)')
ax.semilogx(k_nDGP_avg, Pk_nDGP_Me_interp - Pk_nDGP_avg, lw=2, ls='--', label='nDGP (Our - mean)')
ax.axhline(0, color='k', lw=1, ls=':')
#ax.set_title('Power Spectrum Difference at z=1.65', fontsize=14)
ax.set_xlabel(r'$k \; [h/\mathrm{Mpc}]$', fontsize=14)
ax.set_ylabel(r'$P_{\mathrm{our}}-P_{\mathrm{mean}}$', fontsize=14)
ax.tick_params(axis='both', which='major', labelsize=12)
ax.tick_params(axis='both', which='minor', labelsize=10)
ax.legend()
ax.grid(True, which='both', ls=':', alpha=0.4)
plt.tight_layout()
plt.show()